# Notebook 03: Self-Attention

The core mechanism of transformers. Contents:
- Scaled dot-product attention (forward + backward)
- Causal masking (for autoregressive generation)
- Attention weight visualization
- Gradient checking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Scaled Dot-Product Attention

Given queries $Q$, keys $K$, and values $V$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- $Q, K, V$ are projected from the input: $Q = XW_Q$, $K = XW_K$, $V = XW_V$
- $d_k$ is the key dimension (scaling prevents softmax saturation)
- The softmax produces **attention weights**: how much each position attends to every other position

In [ ]:
def softmax(x):
    """Numerically stable softmax along last axis."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

In [ ]:
class SingleHeadAttention:
    """Single-head scaled dot-product attention with causal mask."""
    
    def __init__(self, d_model, d_k):
        scale = np.sqrt(2.0 / (d_model + d_k))
        self.W_Q = np.random.randn(d_model, d_k) * scale
        self.W_K = np.random.randn(d_model, d_k) * scale
        self.W_V = np.random.randn(d_model, d_k) * scale
        
        self.d_k = d_k
        
        # Gradients
        self.dW_Q = None
        self.dW_K = None
        self.dW_V = None
        
        # Cache for backward
        self.x = None
        self.Q = None
        self.K = None
        self.V = None
        self.attn_weights = None
        self.scores = None
    
    def forward(self, x, mask=None):
        """Forward pass.
        
        Args:
            x: input of shape (batch, seq_len, d_model)
            mask: boolean mask, True = position to MASK (set to -inf)
        
        Returns:
            output of shape (batch, seq_len, d_k)
        """
        self.x = x
        B, T, D = x.shape
        
        # Project to Q, K, V
        self.Q = x @ self.W_Q  # (B, T, d_k)
        self.K = x @ self.W_K
        self.V = x @ self.W_V
        
        # Scaled dot-product
        self.scores = (self.Q @ self.K.transpose(0, 2, 1)) / np.sqrt(self.d_k)  # (B, T, T)
        
        # Apply causal mask
        if mask is not None:
            self.scores = np.where(mask, -1e9, self.scores)
        
        # Softmax to get attention weights
        self.attn_weights = softmax(self.scores)  # (B, T, T)
        
        # Weighted sum of values
        output = self.attn_weights @ self.V  # (B, T, d_k)
        return output
    
    def backward(self, dout):
        """Backward pass through attention.
        
        Args:
            dout: gradient of shape (B, T, d_k)
        Returns:
            dx: gradient w.r.t. input, shape (B, T, d_model)
        """
        B, T, d_k = dout.shape
        
        # Gradient through: output = attn_weights @ V
        d_attn_weights = dout @ self.V.transpose(0, 2, 1)  # (B, T, T)
        dV = self.attn_weights.transpose(0, 2, 1) @ dout   # (B, T, d_k)
        
        # Gradient through softmax
        # d_scores[i,j] = sum_k (d_attn[i,k] * attn[i,k] * (delta_jk - attn[i,j]))
        # Simplified: d_scores = attn * (d_attn - sum(d_attn * attn, axis=-1, keepdims))
        sum_term = np.sum(d_attn_weights * self.attn_weights, axis=-1, keepdims=True)
        d_scores = self.attn_weights * (d_attn_weights - sum_term)  # (B, T, T)
        d_scores /= np.sqrt(self.d_k)
        
        # Gradient through Q @ K^T
        dQ = d_scores @ self.K              # (B, T, d_k)
        dK = d_scores.transpose(0, 2, 1) @ self.Q  # (B, T, d_k)
        
        # Gradient through projections
        x_flat = self.x.reshape(-1, self.x.shape[-1])  # (B*T, d_model)
        self.dW_Q = x_flat.T @ dQ.reshape(-1, d_k)
        self.dW_K = x_flat.T @ dK.reshape(-1, d_k)
        self.dW_V = x_flat.T @ dV.reshape(-1, d_k)
        
        # Gradient through input
        dx = dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T
        return dx

print("SingleHeadAttention defined.")

## 2. Causal Mask

For autoregressive models (like GPT), each position can only attend to **previous** positions (and itself). We achieve this with a causal mask that sets future positions to $-\infty$ before softmax.

In [ ]:
def causal_mask(seq_len):
    """Create causal (upper triangular) mask.
    
    Returns:
        mask: boolean array of shape (1, seq_len, seq_len)
              True = position to MASK (future positions)
    """
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask[np.newaxis, :, :]  # add batch dimension

# Visualize the mask
T = 8
mask = causal_mask(T)
print(f"Causal mask shape: {mask.shape}")
print(f"\nMask (True = blocked):")
print(mask[0].astype(int))

plt.figure(figsize=(5, 4))
plt.imshow(~mask[0], cmap='Blues')  # invert: show allowed positions
plt.xlabel('Key Position (attending to)')
plt.ylabel('Query Position (attending from)')
plt.title('Causal Mask (blue = allowed)')
plt.tight_layout()
plt.show()
print("Each position can only attend to itself and earlier positions.")

## 3. Testing Attention

In [ ]:
# Test with random input
d_model = 64
d_k = 16
B, T = 2, 8  # batch size, sequence length

attn = SingleHeadAttention(d_model, d_k)
x = np.random.randn(B, T, d_model)
mask = causal_mask(T)

# Forward
output = attn.forward(x, mask=mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")

# Backward
dout = np.random.randn(B, T, d_k)
dx = attn.backward(dout)
print(f"dx shape:     {dx.shape}")

## 4. Visualizing Attention Weights

Attention pattern visualization. With the causal mask, attention weights are zero above the diagonal.

In [ ]:
# Visualize attention weights for first sample in batch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Without mask
out_nomask = attn.forward(x, mask=None)
axes[0].imshow(attn.attn_weights[0], cmap='hot')
axes[0].set_title('Attention Weights (no mask)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# With causal mask
out_masked = attn.forward(x, mask=mask)
axes[1].imshow(attn.attn_weights[0], cmap='hot')
axes[1].set_title('Attention Weights (causal mask)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')

plt.tight_layout()
plt.show()
print("With causal mask, attention to future positions is zero.")

## 5. Gradient Check for Attention

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """Compute numerical gradient using central differences."""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]
        x[idx] = old_val + eps
        fp = f(x)
        x[idx] = old_val - eps
        fm = f(x)
        grad[idx] = (fp - fm) / (2 * eps)
        x[idx] = old_val
        it.iternext()
    return grad

def relative_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(np.abs(a) + np.abs(b), 1e-8)))

# Gradient check for W_Q
np.random.seed(42)
d_model, d_k = 8, 4  # small dims for fast checking
B, T = 2, 4
attn = SingleHeadAttention(d_model, d_k)
x = np.random.randn(B, T, d_model)
mask = causal_mask(T)
dout = np.random.randn(B, T, d_k)

# Analytical gradients
output = attn.forward(x, mask=mask)
dx = attn.backward(dout)

# Check W_Q gradient
def loss_wq(W_Q):
    attn.W_Q = W_Q
    out = attn.forward(x, mask=mask)
    return np.sum(out * dout)

num_grad = numerical_gradient(loss_wq, attn.W_Q.copy())
err = relative_error(attn.dW_Q, num_grad)
print(f"W_Q gradient error: {err:.2e} {'PASS' if err < 1e-5 else 'FAIL'}")

# Check W_K gradient
def loss_wk(W_K):
    attn.W_K = W_K
    out = attn.forward(x, mask=mask)
    return np.sum(out * dout)

num_grad = numerical_gradient(loss_wk, attn.W_K.copy())
err = relative_error(attn.dW_K, num_grad)
print(f"W_K gradient error: {err:.2e} {'PASS' if err < 1e-5 else 'FAIL'}")

# Check W_V gradient
def loss_wv(W_V):
    attn.W_V = W_V
    out = attn.forward(x, mask=mask)
    return np.sum(out * dout)

num_grad = numerical_gradient(loss_wv, attn.W_V.copy())
err = relative_error(attn.dW_V, num_grad)
print(f"W_V gradient error: {err:.2e} {'PASS' if err < 1e-5 else 'FAIL'}")

# Check input gradient
def loss_x(x_in):
    out = attn.forward(x_in, mask=mask)
    return np.sum(out * dout)

num_grad = numerical_gradient(loss_x, x.copy())
err = relative_error(dx, num_grad)
print(f"x gradient error:   {err:.2e} {'PASS' if err < 1e-5 else 'FAIL'}")

## Summary

We've built:
- **Scaled dot-product attention** with full forward and backward passes
- **Causal masking** for autoregressive (decoder-only) generation
- **Attention visualization** showing the patterns
- **Gradient checking** to verify correctness

Next notebook: **Multi-Head Attention** - running multiple attention heads in parallel for richer representations.